# Assignment: When Chat Systems Break - A Realistic Sharding Simulation


---
## 📅 Day 9 (9 April): Stress Simulation + Failure Simulation

In [ ]:
import random

class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content


class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []

    def store(self, message):
        self.messages.append(message)

    def count(self):
        return len(self.messages)


class ShardManager:
    def __init__(self, num_shards):
        self.shards = [Shard(i) for i in range(num_shards)]

    def send_message(self, message):
        # will be implemented by subclasses
        pass

    def print_stats(self):
        total = sum(s.count() for s in self.shards)
        print(f"{'Shard':<10} {'Messages':<15} {'% of Total':<10}")
        print('-' * 35)
        for shard in self.shards:
            pct = round((shard.count() / total * 100), 1) if total > 0 else 0
            print(f"Shard {shard.id:<5} {shard.count():<15} {pct}%")
        print(f"{'Total':<10} {total}")

import hashlib


class UserShardManager(ShardManager):
    def get_shard(self, user_id):
        return self.shards[user_id % len(self.shards)]

    def send_message(self, message):
        shard = self.get_shard(message.user_id)
        shard.store(message)


class ChannelShardManager(ShardManager):
    def get_shard(self, channel_id):
        return self.shards[channel_id % len(self.shards)]

    def send_message(self, message):
        shard = self.get_shard(message.channel_id)
        shard.store(message)


class HashShardManager(ShardManager):
    def get_shard(self, key):
        h = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
        return self.shards[h % len(self.shards)]

    def send_message(self, message):
        shard = self.get_shard(message.channel_id)
        shard.store(message)


In [ ]:
# =============================================
# DAY 9: Stress Simulation
# Run all 3 strategies under 3 scenarios:
#   1. Normal day
#   2. Viral event
#   3. Extreme spike
# Also: Hotspot detection, Failure simulation
# =============================================

def simulate(manager, num_users, num_messages, viral_channel=None, viral_ratio=0.0):
    """
    Sends messages to a shard manager.
    viral_channel: if set, this channel gets viral_ratio % of traffic
    """
    viral_count = int(num_messages * viral_ratio)
    normal_count = num_messages - viral_count

    # Viral messages
    if viral_channel:
        for _ in range(viral_count):
            msg = Message(user_id=random.randint(1, num_users),
                          channel_id=viral_channel,
                          content="hello")
            manager.send_message(msg)

    # Normal messages
    for _ in range(normal_count):
        msg = Message(user_id=random.randint(1, num_users),
                      channel_id=random.randint(1, 50),
                      content="hello")
        manager.send_message(msg)


def check_hotspot(manager, threshold=50):
    """Warn if any shard has more than threshold% of total messages"""
    total = sum(s.count() for s in manager.shards)
    if total == 0:
        return
    for shard in manager.shards:
        pct = shard.count() / total * 100
        if pct > threshold:
            print(f"  ⚠️  HOTSPOT WARNING: Shard {shard.id} has {round(pct,1)}% of load! (>{threshold}% threshold)")


scenarios = [
    {"name": "Normal Day",     "users": 1000, "messages": 5000,  "viral_ch": None, "viral_ratio": 0.0},
    {"name": "Viral Event",    "users": 5000, "messages": 20000, "viral_ch": 3,    "viral_ratio": 0.8},
    {"name": "Extreme Spike",  "users": 50000,"messages": 50000, "viral_ch": 3,    "viral_ratio": 0.95},
]

strategies = [
    ("User-Based",    UserShardManager),
    ("Channel-Based", ChannelShardManager),
    ("Hash-Based",    HashShardManager),
]

for scenario in scenarios:
    print(f"\n{'='*55}")
    print(f"SCENARIO: {scenario['name']}")
    print(f"  Users: {scenario['users']} | Messages: {scenario['messages']}")
    if scenario['viral_ch']:
        print(f"  Viral Channel: {scenario['viral_ch']} ({int(scenario['viral_ratio']*100)}% of traffic)")
    print(f"{'='*55}")

    for strategy_name, ManagerClass in strategies:
        mgr = ManagerClass(num_shards=3)
        simulate(mgr,
                 num_users=scenario['users'],
                 num_messages=scenario['messages'],
                 viral_channel=scenario['viral_ch'],
                 viral_ratio=scenario['viral_ratio'])

        print(f"\n  Strategy: {strategy_name}")
        total = sum(s.count() for s in mgr.shards)
        for shard in mgr.shards:
            pct = round(shard.count() / total * 100, 1) if total > 0 else 0
            bar = '█' * int(pct / 5)
            print(f"    Shard {shard.id}: {shard.count():>6} msgs ({pct}%) {bar}")
        check_hotspot(mgr, threshold=50)

In [ ]:
# =============================================
# DAY 9 (continued): Shard Failure Simulation
# Disable one shard and observe what happens
# =============================================

print("=== Failure Simulation: What happens when Shard 0 goes down? ===")
print()

class ShardWithFailure(Shard):
    def __init__(self, shard_id):
        super().__init__(shard_id)
        self.is_down = False   # by default it's working

    def store(self, message):
        if self.is_down:
            # Message is LOST — shard is down
            return False
        self.messages.append(message)
        return True


class FailureChannelManager(ChannelShardManager):
    def __init__(self, num_shards):
        # Override to use failure-aware shards
        self.shards = [ShardWithFailure(i) for i in range(num_shards)]
        self.lost_messages = 0

    def send_message(self, message):
        shard = self.get_shard(message.channel_id)
        success = shard.store(message)
        if not success:
            self.lost_messages += 1


fail_mgr = FailureChannelManager(num_shards=3)

# Send 3000 messages (normal)
for i in range(3000):
    msg = Message(user_id=random.randint(1, 1000),
                  channel_id=random.randint(1, 9),
                  content="hello")
    fail_mgr.send_message(msg)

print("Before failure:")
for shard in fail_mgr.shards:
    print(f"  Shard {shard.id}: {shard.count()} messages, Status: {'✓ UP' if not shard.is_down else '✗ DOWN'}")

# NOW: Shard 0 goes down!
print()
print(">>> Shard 0 has crashed! <<<")
fail_mgr.shards[0].is_down = True

# Send 2000 more messages
for i in range(2000):
    msg = Message(user_id=random.randint(1, 1000),
                  channel_id=random.randint(1, 9),
                  content="hello after crash")
    fail_mgr.send_message(msg)

print()
print("After failure:")
for shard in fail_mgr.shards:
    print(f"  Shard {shard.id}: {shard.count()} messages, Status: {'✓ UP' if not shard.is_down else '✗ DOWN'}")

print()
print(f"Messages LOST because Shard 0 was down: {fail_mgr.lost_messages}")
print("Observation: Any channel that maps to Shard 0 lost all its new messages.")
print("In a real system, we'd need REPLICATION (backup copies) to prevent data loss.")